# 04 — Parallel Agents: Fan-Out and Fan-In

        **Mode:** `offline`  
        **Version:** Praval `0.8.0`  
        **Course link:** [Watch the original course video](https://www.youtube.com/watch?v=VNZ_lhljPBc)

        This notebook makes the framework's execution visible. It records actual
        stages and renders the messages or runtime events that connect them.

        **You will see**

        - One Spore fans out to independent specialists.
- A correlation ID joins asynchronous results.
- A guarded combiner emits one final Spore.

        Run the cells from top to bottom. Committed notebooks contain no saved
        output, so everything shown is produced by your installed Praval package.


In [ ]:
from __future__ import annotations

import html as _html
import json
import os
import time
from pathlib import Path

from IPython.display import HTML, display

os.environ.setdefault("PRAVAL_OBSERVABILITY", "off")


from praval import get_provider_registry
from praval.models import ModelResponse, ProviderCapabilities


class NotebookLifecycleProvider:
    """Credential-free adapter for agents whose handlers do not call a model."""

    provider_name = "notebook-lifecycle"
    capabilities = ProviderCapabilities(text=True)

    def __init__(self, config):
        self.config = config

    def invoke(self, request, tools=None):
        return ModelResponse(
            content="Notebook lifecycle response",
            provider=self.provider_name,
            model=request.model,
        )

    def close(self):
        return None


_notebook_registry = get_provider_registry()
_notebook_registry.register_provider(
    "notebook-lifecycle",
    NotebookLifecycleProvider,
    default_model="notebook-lifecycle-model",
    capabilities=NotebookLifecycleProvider.capabilities,
)
os.environ.setdefault("PRAVAL_DEFAULT_PROVIDER", "notebook-lifecycle")
os.environ.setdefault("PRAVAL_DEFAULT_MODEL", "notebook-lifecycle-model")

EVENTS = []
_STARTED = time.perf_counter()


def stage(actor, action, detail="", spore=None):
    """Capture one observable execution stage."""
    EVENTS.append(
        {
            "ms": round((time.perf_counter() - _STARTED) * 1000, 1),
            "actor": actor,
            "action": action,
            "detail": detail,
            "spore_id": getattr(spore, "id", "") if spore else "",
        }
    )


def show_flow(*steps):
    cards = []
    for index, (name, detail) in enumerate(steps):
        if index:
            cards.append('<div style="font-size:24px;color:#557">→</div>')
        cards.append(
            '<div style="padding:12px 16px;border:1px solid #b8c7dc;'
            'border-radius:12px;background:#f7fbff;min-width:130px">'
            f'<b>{_html.escape(name)}</b><br>'
            f'<span style="color:#456;font-size:12px">'
            f'{_html.escape(detail)}</span></div>'
        )
    display(
        HTML(
            '<div style="display:flex;gap:10px;align-items:center;'
            'flex-wrap:wrap;margin:12px 0">' + "".join(cards) + "</div>"
        )
    )


def show_timeline(events=None):
    events = EVENTS if events is None else events
    rows = []
    for item in events:
        rows.append(
            "<tr>"
            f"<td>{item['ms']:.1f}</td>"
            f"<td><b>{_html.escape(str(item['actor']))}</b></td>"
            f"<td>{_html.escape(str(item['action']))}</td>"
            f"<td>{_html.escape(str(item['detail']))}</td>"
            f"<td><code>{_html.escape(str(item['spore_id'])[:12])}</code></td>"
            "</tr>"
        )
    display(
        HTML(
            '<table style="border-collapse:collapse;width:100%">'
            '<thead><tr><th>ms</th><th>actor</th><th>stage</th>'
            '<th>detail</th><th>spore</th></tr></thead><tbody>'
            + "".join(rows)
            + "</tbody></table>"
        )
    )


def show_spore(spore, label="Spore on the Reef"):
    payload = json.loads(spore.to_json())
    rendered = _html.escape(json.dumps(payload, indent=2, sort_keys=True))
    display(
        HTML(
            '<div style="border-left:5px solid #7b61ff;padding:10px 14px;'
            'background:#faf9ff;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def show_json(value, label="Result"):
    rendered = _html.escape(json.dumps(value, indent=2, sort_keys=True, default=str))
    display(
        HTML(
            '<div style="border-left:5px solid #18a999;padding:10px 14px;'
            'background:#f5fffd;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def require_env(*names):
    missing = [name for name in names if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing required notebook configuration: " + ", ".join(missing))
    return {name: os.environ[name] for name in names}


def find_example_asset(relative):
    relative = Path(relative)
    starts = [Path.cwd(), *Path.cwd().parents]
    for root in starts:
        for candidate in (root / relative, root / "examples" / relative):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Could not locate example asset: {relative}")


## One input, three concurrent handlers

The analyzers deliberately finish at different times. The timeline makes the
concurrency visible, while the session ID prevents results from different
requests from being mixed.


In [ ]:
show_flow(
    ("Input Spore", "session-42"),
    ("Three analyzers", "fan-out"),
    ("Combiner", "join by session"),
    ("Final Spore", "fan-in"),
)


In [ ]:
import threading

from praval import agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef

reset_reef()
session_results = {}
combined_sessions = set()
final_results = []
result_spores = []
result_lock = threading.Lock()


def emit_analysis(spore, analyzer, result, delay):
    time.sleep(delay)
    stage(analyzer, "analysis complete", str(result), spore)
    broadcast(
        {
            "type": "analysis_result",
            "session_id": spore.knowledge["session_id"],
            "analyzer": analyzer,
            "result": result,
        }
    )


@agent("course-sentiment", responds_to=["analyze"], auto_broadcast=False)
def sentiment(spore):
    emit_analysis(spore, "sentiment", "positive", 0.08)


@agent("course-keywords", responds_to=["analyze"], auto_broadcast=False)
def keywords(spore):
    words = [word.strip(".,!") for word in spore.knowledge["text"].split()]
    emit_analysis(spore, "keywords", words[-3:], 0.03)


@agent("course-length", responds_to=["analyze"], auto_broadcast=False)
def length(spore):
    emit_analysis(spore, "length", len(spore.knowledge["text"].split()), 0.05)


@agent("course-combiner", responds_to=["analysis_result"], auto_broadcast=False)
def combiner(spore):
    result_spores.append(spore)
    session = spore.knowledge["session_id"]
    with result_lock:
        bucket = session_results.setdefault(session, {})
        bucket[spore.knowledge["analyzer"]] = spore.knowledge["result"]
        ready = len(bucket) == 3 and session not in combined_sessions
        if ready:
            combined_sessions.add(session)
            combined = dict(bucket)
    stage("combiner", "result received", spore.knowledge["analyzer"], spore)
    if ready:
        broadcast(
            {
                "type": "combined_analysis",
                "session_id": session,
                "analysis": combined,
            }
        )


@agent("course-parallel-output", responds_to=["combined_analysis"], auto_broadcast=False)
def final_output(spore):
    final_results.append(spore.knowledge)
    stage("output", "fan-in complete", spore.knowledge["session_id"], spore)


In [ ]:
start_agents(
    sentiment,
    keywords,
    length,
    combiner,
    final_output,
    initial_data={
        "type": "analyze",
        "session_id": "session-42",
        "text": "Praval makes concurrent agent behavior visible and composable.",
    },
    channel="course-parallel",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=10)
assert len(final_results) == 1
assert set(final_results[0]["analysis"]) == {"sentiment", "keywords", "length"}


In [ ]:
show_json(final_results[0], "Joined result")
show_spore(result_spores[0], "One parallel result Spore")
show_timeline()


In [ ]:
for function in (sentiment, keywords, length, combiner, final_output):
    function._praval_agent.close()
assert reef.shutdown(timeout=10)
